In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType
from pyspark.sql import functions as F

In [0]:
spark=SparkSession.builder.appName("BigMartSales").getOrCreate()
df=spark.table("default.big_mart_sales")

In [0]:
df.show(5)

In [0]:
display(df.limit(5))

In [0]:
json_df=spark.table("default.drivers")


In [0]:
display(json_df.limit(5))

In [0]:
df.printSchema()

In [0]:

my_ddl_schema = '''
                    Item_Identifier STRING,
                    Item_Weight STRING,
                    Item_Fat_Content STRING, 
                    Item_Visibility DOUBLE,
                    Item_Type STRING,
                    Item_MRP DOUBLE,
                    Outlet_Identifier STRING,
                    Outlet_Establishment_Year INT,
                    Outlet_Size STRING,
                    Outlet_Location_Type STRING, 
                    Outlet_Type STRING,
                    Item_Outlet_Sales DOUBLE 

                ''' 


schema=StructType.fromDDL(my_ddl_schema)
df2=spark.table("default.big_mart_sales")

In [0]:
df=df2.select(*[F.col(field.name).cast(field.dataType).alias(field.name) for field in schema.fields])
df.printSchema()

In [0]:
from pyspark.sql.functions import col
df.select(col("Item_Identifier"),col("Item_Weight"),col("Item_Fat_Content")).display()


In [0]:
df.select(col("Item_Identifier").alias("Item_ID")).display()

In [0]:
df=df.withColumnRenamed("Item_Identifier","Item_ID")

In [0]:
display(df.limit(5))

In [0]:
df.filter(col("Item_Fat_Content")=="Low Fat").display()

In [0]:
df.filter((col("Item_Type")=="Frozen Foods") & (col("Outlet_Type")=="Supermarket Type1")).display()

In [0]:
from pyspark.sql.functions import lit
df=df.withColumn("New_Feature",lit("hello"))

In [0]:
df.withColumn("New_Feature",col("Item_Weight")*col("Item_MRP")).display()

In [0]:
from pyspark.sql.functions import regexp_replace, col
df=df.withColumn("Item_Fat_Content",regexp_replace(col("Item_Fat_Content"),"Regular","Reg"))
df=df.withColumn("Item_Fat_Content",regexp_replace(col("Item_Fat_Content"),"Low Fat","LF"))

In [0]:
from pyspark.sql.types import StringType, DoubleType, IntegerType

df=df.withColumn("Item_Weight",col("Item_Weight").cast(StringType()))


In [0]:
df.printSchema()

In [0]:
df.sort(col("Item_Weight").desc()).display()

In [0]:
df.sort(col("Item_Visibility").asc()).display()

In [0]:
df.sort(["Item_Weight","Item_Visibility"],ascending=[False,True]).display()

In [0]:
df.sort(["Item_Weight","Item_Visibility"],ascending=[0,0]).display()

In [0]:
df.drop("Item_ID").display()

In [0]:
df.dropDuplicates().display()

In [0]:

df.distinct().display()
     

In [0]:
from pyspark.sql.functions import col, sum, when

df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]).display()

In [0]:
data_1=[("1","kid"),
        ("2","people"),
        ("3","children"),
        ("4","adult"),]
schema_1="id string, name string"

df3=spark.createDataFrame(data=data_1,schema=schema_1)
df3.display()


In [0]:
data2=[("1","kid"),
        ("2","people"),
        ("3","children"),
        ("4","sky")]
schema="id string, name string"

df4=spark.createDataFrame(data=data2,schema=schema)
df4.display()

In [0]:
df3.union(df4).display()

# Initcap

In [0]:
from pyspark.sql.functions import upper
df.select(upper("Item_Type").alias("upper_Item_Type")).display()

In [0]:
from pyspark.sql.functions import current_date,date_add,date_sub,date_diff,date_format
df = df.withColumn('curr_date', current_date())


In [0]:
df = df.withColumn('week_after',date_add('curr_date',7))

In [0]:
display(df.limit(5))

In [0]:
df.withColumn("week_before",date_sub("curr_date",7)).display()

In [0]:
df.withColumn("datediff",date_diff("week_after","curr_date")).display()

In [0]:
df.withColumn("week_before",date_format("week_before","dd-MM-yyyy"))

In [0]:

df.dropna('all').display()

In [0]:
df.fillna("Not Available").display()

# Split

In [0]:
from pyspark.sql.functions import split
df.withColumn("Outlet_Type",split("Outlet_Type"," ")).display()

In [0]:
df.withColumn("Outlet_Type",split("Outlet_Type"," ")[1]).display()

In [0]:
df_ex=df.withColumn("Outlet_Type",split("Outlet_Type"," "))
display(df_ex.limit(5))

In [0]:
df.groupBy("Item_Type").agg(sum("Item_MRP")).display()

In [0]:
df.groupby("Item_Type","Outlet_Size").agg(sum("Item_MRP").alias("Total_MRP")).display()

In [0]:
from pyspark.sql.functions import avg, sum
df.groupBy("Item_Type").pivot("Outlet_Size").agg(avg("Item_MRP")).display()

In [0]:
df=df.withColumn("veg_flag",when(col("Item_Type")=="Meat","Non-Veg").otherwise("Veg"))

In [0]:
df.display()

In [0]:
data1=[("1","cat","hello"),
     ("2","dog","world"),
     ("3","mouse","databricks"),
     ("4","horse","spark"),
     ("5","cow","python"),
     ("6","pig","scala")]
schema_1="id string, animal string, animal_id string"

df1=spark.createDataFrame(data=data1,schema=schema_1)
df1.display()


In [0]:
data_2=[("1","cat"),
     ("2","dog"),
     ("3","mouse"),
     ("4","horse"),
     ("5","cow")]
schema_2="animal_id string, animal string"

df2=spark.createDataFrame(data=data_2,schema=schema_2)
df2.display()


In [0]:

df1.join(df2, df1['animal_id']==df2['animal_id'],'inner').display()

In [0]:
df1.join(df2,df1["animal_id"]==df2["animal_id"],"left").display()


In [0]:
df1.join(df2,df1["animal_id"]==df2["animal_id"],"right").display()

In [0]:
df1.join(df2,df1["animal_id"]==df2["animal_id"],"anti").display()

In [0]:
display(df.limit(5))

In [0]:
from pyspark.sql.window import Window

In [0]:
from pyspark.sql.functions import row_number
windowSpec = Window.orderBy("Item_ID")

df_with_rn = df.withColumn("rowCol", row_number().over(windowSpec))

df_with_rn.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank

# Example window spec
windowSpec = Window.partitionBy("Outlet_Identifier").orderBy("Item_MRP")

# Row number: strict sequential numbering
df_rn = df.withColumn("row_number", row_number().over(windowSpec))

# Rank: numbers with gaps if there are ties
df_rk = df_rn.withColumn("rank", rank().over(windowSpec))

# Dense rank: no gaps if there are ties
df_final = df_rk.withColumn("dense_rank", dense_rank().over(windowSpec))

df_final.display()
